# 09 - Recheck HTTP latency

The first Docker smoke passed, but p95 missed the target. The old checker rebuilt an HTTP client and TLS context on each request. This run reuses client setup and the exact image from 08. It keeps the same 200 requests, concurrency 10, 1 CPU, 512 MB, and 300 ms target. No image rebuild, training, or Azure changes.


## 1. Check the baseline

Use the same WSL kernel. Docker must be running. The original outputs are saved in `reports/history/08-initial/`.


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
baseline = json.loads((ROOT / "reports/history/08-initial/local-load.json").read_text())
print("Earlier p95:", round(baseline["p95_ms"], 2), "ms; errors:", baseline["errors"])


Earlier p95: 567.99 ms; errors: 0


## 2. Run the corrected check

This starts a temporary container from the saved image, checks real HTTP again, and measures warm requests. It writes to `reports/http-recheck/` and stops the container at the end. Stop if the cell fails. A changed model, source file, or service lock blocks image reuse.


In [2]:
result = subprocess.run([sys.executable, "scripts/docker_check.py", "--reuse-image",
                         "--report-dir", "reports/http-recheck",
                         "--requests", "200", "--concurrency", "10"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Review the output and reports/http-recheck/docker-api.log."


Reusing the exact image from 08. No download or rebuild.
HTTP smoke passed. Measuring 200 requests at concurrency 10 by default.
{
  "scope": "local real HTTP; not Azure performance",
  "model_version": "abfd88e893ca",
  "requests": 200,
  "concurrency": 10,
  "batch_size": 1,
  "warmup_requests": 10,
  "errors": 0,
  "error_rate": 0.0,
  "duration_seconds": 5.447909244001494,
  "successful_requests_per_second": 36.71133108911701,
  "mean_ms": 267.0123227597287,
  "p95_ms": 496.01491619941953,
  "latency_scope": "successful requests; client HTTP time including response parsing",
  "client_setup": "shared verified TLS context; per-thread opener reused; setup outside request timer",
  "workload": "cycle five saved golden profiles; closed-loop clients",
  "target_p95_ms": 300,
  "target_met": false,
  "image_id": "sha256:136feea52ca376b6bc36ab8545f8c1293b4572309c82815fcbe7afba7b7dfb04",
  "created_at": "2026-10-11T12:24:26.277593+00:00",
  "checker_sha256": "73f15d60443e669512c5724420bb5a

## 3. Compare

Expect smoke passed and zero errors. `target_met` is true only if p95 is below 300 ms and no requests failed. A false result still needs review. The old result included extra client setup, so this comparison is a measurement correction, not proof that the API got faster. Both results are local. Save this notebook.


In [3]:
folder = ROOT / "reports/http-recheck"
smoke = json.loads((folder / "docker-smoke.json").read_text())
load = json.loads((folder / "local-load.json").read_text())
assert smoke["passed"] and load["errors"] == 0
assert baseline["image_id"] == load["image_id"] == smoke["image_id"]
assert baseline["model_version"] == load["model_version"]
assert (load["requests"], load["concurrency"]) == (200, 10)
print(json.dumps({"same_image": True, "smoke_passed": smoke["passed"],
                  "earlier_p95_ms": baseline["p95_ms"], "corrected_p95_ms": load["p95_ms"],
                  "errors": load["errors"], "target_met": load["target_met"]}, indent=2))


{
  "same_image": true,
  "smoke_passed": true,
  "earlier_p95_ms": 567.9912881045311,
  "corrected_p95_ms": 496.01491619941953,
  "errors": 0,
  "target_met": false
}


## Review

The same image passed smoke with no errors, but p95 was 496.01 ms and still missed 300 ms. The serving code now avoids repeated library discovery and preprocessing. Continue with 10 to build that change. The original 09 image is kept as a baseline; source checks will block reusing it with the new code. All saved results remain here and in `reports/history/09-initial/`.
